In [0]:
select * from com_edp_prd.com_raw.kom_providers limit 1


In [0]:
WITH hcp_list AS (
  SELECT * FROM (
    VALUES
      ('Jordan Zieger'),
      ('Sam Seymour'),
      ('Kevin Kues'),
      ('Kara Woolgar'),
      ('Frankie Burney'),
      ('Ashley Glenn'),
      ('Tim Wood'),
      ('Bobbie Salveson'),
      ('Jeff Posadas'),
      ('Teresa Hahn'),
      ('Zoey Trueblood'),
      ('Diego'),
      ('Theresa Gonzales'),
      ('Kathryn Gasperian'),
      ('Bex Jaeger'),
      ('Jennifer Widmer'),
      ('Madeleine Ciobanu'),
      ('Angela Campbell'),
      ('Arthur Lanahan'),
      ('Drea Petersen'),
      ('Ari Nouraee'),
      ('Katherine Kim'),
      ('Rachel Hickley'),
      ('Olivia Brown'),
      ('David Stefanoni'),
      ('Queenie Tan'),
      ('Elizabeth Null'),
      ('Lauren Pfeil'),
      ('Ghada Hiljazi'),
      ('Bryan Hainline'),
      ('Jackie Beirne'),
      ('Colleen Dansereau'),
      ('Lauren O''Grady'),
      ('Jolie Matheson'),
      ('Shannon Dixon'),
      ('Sharon Anderson'),
      ('Alejandra Gomez'),
      ('Lorien King'),
      ('Cheryl Clow'),
      ('Berrin Montreleone'),
      ('Jillian Soler'),
      ('Melissa Byler'),
      ('Wing Hung'),
      ('Jennifer Baker'),
      ('Deepa Rajan'),
      ('Grace Meier'),
      ('Todd Snyder'),
      ('Christine Giummo'),
      ('Monica Borden'),
      ('Amanda Prince'),
      ('Amarillis Sanchez-Valle'),
      ('Amy Pesky'),
      ('Lauren Wilson'),
      ('Erin Kissenberg'),
      ('Paulo Mendoza'),
      ('Clay Ferren'),
      ('Kathryn Dempsey'),
      ('Kaleigh Bright'),
      ('Sarah Young'),
      ('Deeksha Bali'),
      ('Joe Muenzer'),
      ('Kim Stephens'),
      ('Jessica Upshaw')
  ) AS t(full_name)
),
hcp_parsed AS (
  SELECT
    full_name,
    -- first token = first name
    SPLIT_PART(full_name, ' ', 1) AS first_name,
    -- everything after first token = last name (supports O'Grady, Sanchez-Valle, etc.)
    NULLIF(TRIM(SUBSTRING(full_name FROM POSITION(' ' IN full_name) + 1)), '') AS last_name
  FROM hcp_list
)
SELECT
  l.full_name,
  p.npi,
  p.provider_type,
  p.first_name,
  p.last_name,
  p.organization_name,
  p.primary_specialty,
  p.secondary_specialty
FROM hcp_parsed l
JOIN com_edp_prd.com_raw.kom_providers p
  ON (
       l.last_name IS NOT NULL
       AND LOWER(p.first_name) = LOWER(l.first_name)
       AND LOWER(p.last_name)  = LOWER(l.last_name)
     )
  OR (
       l.last_name IS NULL
       AND (
            LOWER(p.first_name) = LOWER(l.first_name)
         OR LOWER(p.last_name)  = LOWER(l.first_name)
       )
     )
ORDER BY l.full_name, p.npi;


In [0]:
%python
#!/usr/bin/env python3
"""
Fetch HCP details from the CMS NPI Registry (public API) by provider name list.

API docs: https://npiregistry.cms.hhs.gov/registry/help-api
Endpoint: https://npiregistry.cms.hhs.gov/api/

Outputs:
- npi_results.csv (flattened, easy to use)
- npi_results.json (raw-ish, full detail)

Usage:
  python npi_lookup.py
"""

from __future__ import annotations

import csv
import json
import time
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import requests


# -----------------------------
# INPUT: Your HCP list
# -----------------------------
HCP_NAMES = [
    "Jordan Zieger",
    "Sam Seymour",
    "Kevin Kues",
    "Kara Woolgar",
    "Frankie Burney",
    "Ashley Glenn",
    "Tim Wood",
    "Bobbie Salveson",
    "Jeff Posadas",
    "Teresa Hahn",
    "Zoey Trueblood",
    "Diego",
    "Theresa Gonzales",
    "Kathryn Gasperian",
    "Bex Jaeger",
    "Jennifer Widmer",
    "Madeleine Ciobanu",
    "Angela Campbell",
    "Arthur Lanahan",
    "Drea Petersen",
    "Ari Nouraee",
    "Katherine Kim",
    "Rachel Hickley",
    "Olivia Brown",
    "David Stefanoni",
    "Queenie Tan",
    "Elizabeth Null",
    "Lauren Pfeil",
    "Ghada Hiljazi",
    "Bryan Hainline",
    "Jackie Beirne",
    "Colleen Dansereau",
    "Lauren O'Grady",
    "Jolie Matheson",
    "Shannon Dixon",
    "Sharon Anderson",
    "Alejandra Gomez",
    "Lorien King",
    "Cheryl Clow",
    "Berrin Montreleone",
    "Jillian Soler",
    "Melissa Byler",
    "Wing Hung",
    "Jennifer Baker",
    "Deepa Rajan",
    "Grace Meier",
    "Todd Snyder",
    "Christine Giummo",
    "Monica Borden",
    "Amanda Prince",
    "Amarillis Sanchez-Valle",
    "Amy Pesky",
    "Lauren Wilson",
    "Erin Kissenberg",
    "Paulo Mendoza",
    "Clay Ferren",
    "Kathryn Dempsey",
    "Kaleigh Bright",
    "Sarah Young",
    "Deeksha Bali",
    "Joe Muenzer",
    "Kim Stephens",
    "Jessica Upshaw",
]


# -----------------------------
# Config
# -----------------------------
NPI_API_URL = "https://npiregistry.cms.hhs.gov/api/"
API_VERSION = "2.1"

# Max results per name query (API supports "limit")
MAX_RESULTS_PER_QUERY = 20

# If too many matches, keep only top N in CSV (JSON will still keep all)
MAX_MATCHES_TO_FLATTEN = 5

# Rate limiting (be nice)
SLEEP_SECONDS_BETWEEN_CALLS = 0.15

# Optional filters you can set:
DEFAULT_STATE_FILTER: Optional[str] = None  # e.g., "CA"
DEFAULT_TAXONOMY_FILTER: Optional[str] = None  # e.g., "207R00000X" (Internal Medicine)


# -----------------------------
# Helpers
# -----------------------------
def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


@dataclass
class ParsedName:
    raw: str
    first: Optional[str]
    last: Optional[str]


def parse_name(full_name: str) -> ParsedName:
    """
    Split into first + last.
    If single token, only first is populated (used for first-name search).
    If multiple tokens, first is token[0], last is everything else.
    """
    name = normalize_spaces(full_name)
    parts = name.split(" ")
    if len(parts) == 1:
        return ParsedName(raw=full_name, first=parts[0], last=None)
    first = parts[0]
    last = " ".join(parts[1:])
    return ParsedName(raw=full_name, first=first, last=last)


def npi_search(
    session: requests.Session,
    first_name: Optional[str],
    last_name: Optional[str],
    state: Optional[str] = None,
    taxonomy: Optional[str] = None,
    limit: int = MAX_RESULTS_PER_QUERY,
) -> Dict[str, Any]:
    """
    Search NPI registry for individual providers (enumeration_type = NPI-1).
    """
    params = {
        "version": API_VERSION,
        "enumeration_type": "NPI-1",
        "limit": limit,
    }
    if first_name:
        params["first_name"] = first_name
    if last_name:
        params["last_name"] = last_name
    if state:
        params["state"] = state
    if taxonomy:
        params["taxonomy_description"] = taxonomy  # alternative: use "taxonomy_description"
        # If you want taxonomy code filtering instead, we can do post-filtering.

    resp = session.get(NPI_API_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


def pick_best_matches(results: List[Dict[str, Any]], parsed: ParsedName) -> List[Dict[str, Any]]:
    """
    Naive ranking:
    - exact first + last match (case-insensitive) gets highest
    - last match only gets next
    - otherwise keep as is
    """
    def score(r: Dict[str, Any]) -> Tuple[int, int]:
        basic = r.get("basic", {}) or {}
        first = (basic.get("first_name") or "").strip().lower()
        last = (basic.get("last_name") or "").strip().lower()
        pf = (parsed.first or "").strip().lower()
        pl = (parsed.last or "").strip().lower()

        if pf and pl and first == pf and last == pl:
            return (3, 0)
        if pl and last == pl:
            return (2, 0)
        if pf and first == pf:
            return (1, 0)
        return (0, 0)

    return sorted(results, key=score, reverse=True)


def flatten_provider(provider: Dict[str, Any], query_name: str, match_rank: int) -> Dict[str, Any]:
    """
    Flatten key fields into a single row.
    Addresses: prefer "LOCATION" and "MAILING" types if present.
    """
    basic = provider.get("basic", {}) or {}
    taxonomies = provider.get("taxonomies", []) or []
    addresses = provider.get("addresses", []) or []
    identifiers = provider.get("identifiers", []) or []

    def find_addr(addr_type: str) -> Optional[Dict[str, Any]]:
        for a in addresses:
            if (a.get("address_purpose") or "").upper() == addr_type.upper():
                return a
        return None

    loc = find_addr("LOCATION")
    mail = find_addr("MAILING")

    primary_tax = next((t for t in taxonomies if t.get("primary") is True), None)
    if primary_tax is None and taxonomies:
        primary_tax = taxonomies[0]

    row = {
        "query_name": query_name,
        "match_rank": match_rank,
        "npi": provider.get("number"),
        "enumeration_date": basic.get("enumeration_date"),
        "last_updated": basic.get("last_updated"),
        "name_prefix": basic.get("name_prefix"),
        "first_name": basic.get("first_name"),
        "middle_name": basic.get("middle_name"),
        "last_name": basic.get("last_name"),
        "name_suffix": basic.get("name_suffix"),
        "credential": basic.get("credential"),
        "gender": basic.get("gender"),
        "sole_proprietor": basic.get("sole_proprietor"),
        "status": basic.get("status"),
        "primary_taxonomy_code": (primary_tax or {}).get("code"),
        "primary_taxonomy_desc": (primary_tax or {}).get("desc"),
        "primary_taxonomy_state": (primary_tax or {}).get("state"),
        "primary_taxonomy_license": (primary_tax or {}).get("license"),
        # Location address
        "loc_address_1": (loc or {}).get("address_1"),
        "loc_address_2": (loc or {}).get("address_2"),
        "loc_city": (loc or {}).get("city"),
        "loc_state": (loc or {}).get("state"),
        "loc_postal_code": (loc or {}).get("postal_code"),
        "loc_country_code": (loc or {}).get("country_code"),
        "loc_phone": (loc or {}).get("telephone_number"),
        "loc_fax": (loc or {}).get("fax_number"),
        # Mailing address
        "mail_address_1": (mail or {}).get("address_1"),
        "mail_address_2": (mail or {}).get("address_2"),
        "mail_city": (mail or {}).get("city"),
        "mail_state": (mail or {}).get("state"),
        "mail_postal_code": (mail or {}).get("postal_code"),
        "mail_country_code": (mail or {}).get("country_code"),
        "mail_phone": (mail or {}).get("telephone_number"),
        "mail_fax": (mail or {}).get("fax_number"),
        # counts / misc
        "address_count": len(addresses),
        "taxonomy_count": len(taxonomies),
        "identifier_count": len(identifiers),
    }
    return row


def main() -> None:
    out_rows: List[Dict[str, Any]] = []
    raw_out: Dict[str, Any] = {}

    with requests.Session() as session:
        # Set a UA
        session.headers.update({"User-Agent": "npi-registry-lookup/1.0"})

        for name in HCP_NAMES:
            if not name or name.strip().upper() == "HCP":
                continue

            parsed = parse_name(name)

            # Primary search: first+last if possible; else first-name only
            try:
                data = npi_search(
                    session=session,
                    first_name=parsed.first,
                    last_name=parsed.last,
                    state=DEFAULT_STATE_FILTER,
                    taxonomy=DEFAULT_TAXONOMY_FILTER,
                )
                time.sleep(SLEEP_SECONDS_BETWEEN_CALLS)
            except Exception as e:
                raw_out[name] = {"error": str(e), "results": []}
                continue

            results = data.get("results", []) or []
            ranked = pick_best_matches(results, parsed)

            raw_out[name] = {
                "query": {"first_name": parsed.first, "last_name": parsed.last},
                "result_count": len(results),
                "results": ranked,
            }

            # Flatten top matches for CSV usability
            for idx, provider in enumerate(ranked[:MAX_MATCHES_TO_FLATTEN], start=1):
                out_rows.append(flatten_provider(provider, query_name=name, match_rank=idx))

            # If no results and it was a single-name entry, try alternate (last_name=token)
            if not results and parsed.last is None and parsed.first:
                try:
                    alt = npi_search(
                        session=session,
                        first_name=None,
                        last_name=parsed.first,
                        state=DEFAULT_STATE_FILTER,
                        taxonomy=DEFAULT_TAXONOMY_FILTER,
                    )
                    time.sleep(SLEEP_SECONDS_BETWEEN_CALLS)
                except Exception as e:
                    raw_out[name]["alt_error"] = str(e)
                    continue

                alt_results = alt.get("results", []) or []
                alt_ranked = pick_best_matches(alt_results, ParsedName(raw=name, first=None, last=parsed.first))

                raw_out[name]["alt_query"] = {"first_name": None, "last_name": parsed.first}
                raw_out[name]["alt_result_count"] = len(alt_results)
                raw_out[name]["alt_results"] = alt_ranked

                for idx, provider in enumerate(alt_ranked[:MAX_MATCHES_TO_FLATTEN], start=1):
                    out_rows.append(flatten_provider(provider, query_name=name, match_rank=idx))

    # Write CSV
    csv_path = "npi_results.csv"
    if out_rows:
        fieldnames = list(out_rows[0].keys())
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(out_rows)

    # Write JSON
    json_path = "npi_results.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(raw_out, f, ensure_ascii=False, indent=2)

    print(f"Done. Wrote {csv_path} and {json_path}")
    print(f"CSV rows: {len(out_rows)} | Names processed: {len([n for n in HCP_NAMES if n.strip() and n.strip().upper()!='HCP'])}")


if __name__ == "__main__":
    main()


In [0]:
%python
import pandas as pd

df = pd.read_csv("npi_results.csv")
display(df)              # Databricks
# or: df.head(20)


In [0]:
%python
#!/usr/bin/env python3
"""
Full NPI registry extractor for a list of HCP names.
Outputs:
 - npi_raw.json         : raw results per query name (everything the API returns)
 - providers.csv        : one row per provider (NPI) with top-level fields; nested arrays kept as JSON strings
 - addresses.csv        : one row per address (linked by npi)
 - taxonomies.csv       : one row per taxonomy (linked by npi)
 - identifiers.csv      : one row per identifier (linked by npi)

Notes:
 - This script queries the public CMS NPI Registry API (no auth required).
 - For single-token names (e.g., "Diego") we first try as first_name then as last_name if no results.
"""

from __future__ import annotations
import csv
import json
import time
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import requests

# -----------------------------
# INPUT: Your HCP list
# -----------------------------
HCP_NAMES = [
    "Jordan Zieger","Sam Seymour","Kevin Kues","Kara Woolgar","Frankie Burney","Ashley Glenn",
    "Tim Wood","Bobbie Salveson","Jeff Posadas","Teresa Hahn","Zoey Trueblood","Diego",
    "Theresa Gonzales","Kathryn Gasperian","Bex Jaeger","Jennifer Widmer","Madeleine Ciobanu",
    "Angela Campbell","Arthur Lanahan","Drea Petersen","Ari Nouraee","Katherine Kim",
    "Rachel Hickley","Olivia Brown","David Stefanoni","Queenie Tan","Elizabeth Null",
    "Lauren Pfeil","Ghada Hiljazi","Bryan Hainline","Jackie Beirne","Colleen Dansereau",
    "Lauren O'Grady","Jolie Matheson","Shannon Dixon","Sharon Anderson","Alejandra Gomez",
    "Lorien King","Cheryl Clow","Berrin Montreleone","Jillian Soler","Melissa Byler",
    "Wing Hung","Jennifer Baker","Deepa Rajan","Grace Meier","Todd Snyder","Christine Giummo",
    "Monica Borden","Amanda Prince","Amarillis Sanchez-Valle","Amy Pesky","Lauren Wilson",
    "Erin Kissenberg","Paulo Mendoza","Clay Ferren","Kathryn Dempsey","Kaleigh Bright",
    "Sarah Young","Deeksha Bali","Joe Muenzer","Kim Stephens","Jessica Upshaw"
]

# -----------------------------
# Config
# -----------------------------
NPI_API_URL = "https://npiregistry.cms.hhs.gov/api/"
API_VERSION = "2.1"
MAX_RESULTS_PER_QUERY = 200   # get as many as the API will return per call (200 is safe)
SLEEP_SECONDS_BETWEEN_CALLS = 0.12

# -----------------------------
# Helpers
# -----------------------------
def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

@dataclass
class ParsedName:
    raw: str
    first: Optional[str]
    last: Optional[str]

def parse_name(full_name: str) -> ParsedName:
    name = normalize_spaces(full_name)
    parts = name.split(" ")
    if len(parts) == 1:
        return ParsedName(raw=full_name, first=parts[0], last=None)
    first = parts[0]
    last = " ".join(parts[1:])
    return ParsedName(raw=full_name, first=first, last=last)

def npi_search(session: requests.Session, first_name: Optional[str], last_name: Optional[str], limit: int = MAX_RESULTS_PER_QUERY) -> Dict[str, Any]:
    params = {"version": API_VERSION, "enumeration_type": "NPI-1", "limit": limit}
    if first_name:
        params["first_name"] = first_name
    if last_name:
        params["last_name"] = last_name
    resp = session.get(NPI_API_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

def flatten_provider_row(provider: Dict[str, Any], query_name: str, match_index: int) -> Dict[str, Any]:
    basic = provider.get("basic", {}) or {}
    taxonomies = provider.get("taxonomies", []) or []
    addresses = provider.get("addresses", []) or []
    identifiers = provider.get("identifiers", []) or []

    # choose a primary taxonomy if marked
    primary_tax = next((t for t in taxonomies if t.get("primary") is True), None)
    if primary_tax is None and taxonomies:
        primary_tax = taxonomies[0]

    row = {
        "query_name": query_name,
        "match_index": match_index,
        "npi": provider.get("number"),
        "enumeration_type": provider.get("enumeration_type"),
        "enumeration_date": basic.get("enumeration_date"),
        "last_updated": basic.get("last_updated"),
        "name_prefix": basic.get("name_prefix"),
        "first_name": basic.get("first_name"),
        "middle_name": basic.get("middle_name"),
        "last_name": basic.get("last_name"),
        "name_suffix": basic.get("name_suffix"),
        "credential": basic.get("credential"),
        "gender": basic.get("gender"),
        "status": basic.get("status"),
        "primary_taxonomy_code": (primary_tax or {}).get("code"),
        "primary_taxonomy_desc": (primary_tax or {}).get("desc"),
        "primary_taxonomy_state": (primary_tax or {}).get("state"),
        # keep nested arrays as JSON strings so CSV still contains all data
        "addresses_json": json.dumps(addresses, ensure_ascii=False),
        "taxonomies_json": json.dumps(taxonomies, ensure_ascii=False),
        "identifiers_json": json.dumps(identifiers, ensure_ascii=False),
        "raw_provider_json": json.dumps(provider, ensure_ascii=False)
    }
    return row

# -----------------------------
# Main
# -----------------------------
def main():
    raw_out: Dict[str, Any] = {}
    providers_rows: List[Dict[str, Any]] = []
    addresses_rows: List[Dict[str, Any]] = []
    taxonomies_rows: List[Dict[str, Any]] = []
    identifiers_rows: List[Dict[str, Any]] = []

    with requests.Session() as session:
        session.headers.update({"User-Agent": "npi-registry-full-export/1.0"})

        for name in HCP_NAMES:
            if not name or not name.strip():
                continue
            parsed = parse_name(name)
            query_attempts = []

            # Attempt 1: first + last if available, else first-only
            try:
                resp = npi_search(session, parsed.first, parsed.last)
                time.sleep(SLEEP_SECONDS_BETWEEN_CALLS)
            except Exception as e:
                raw_out[name] = {"error": str(e), "results": []}
                continue

            results = resp.get("results", []) or []
            # If no results and single token name, try as last_name
            if not results and parsed.last is None and parsed.first:
                try:
                    alt = npi_search(session, None, parsed.first)
                    time.sleep(SLEEP_SECONDS_BETWEEN_CALLS)
                except Exception as e:
                    raw_out[name] = {"error": str(e), "results": []}
                    continue
                results = alt.get("results", []) or []
                raw_out[name] = {
                    "original_query": {"first_name": parsed.first, "last_name": parsed.last},
                    "alt_query": {"first_name": None, "last_name": parsed.first},
                    "result_count": len(results),
                    "results": results
                }
            else:
                raw_out[name] = {
                    "original_query": {"first_name": parsed.first, "last_name": parsed.last},
                    "result_count": len(results),
                    "results": results
                }

            # Export every result (no ranking cap)
            for idx, provider in enumerate(results, start=1):
                # provider-level CSV row (flattened fields + JSON columns)
                providers_rows.append(flatten_provider_row(provider, name, idx))

                # normalized address rows
                for a_idx, addr in enumerate(provider.get("addresses", []) or [], start=1):
                    addresses_rows.append({
                        "query_name": name,
                        "npi": provider.get("number"),
                        "provider_match_index": idx,
                        "address_index": a_idx,
                        "address_purpose": addr.get("address_purpose"),
                        "address_1": addr.get("address_1"),
                        "address_2": addr.get("address_2"),
                        "city": addr.get("city"),
                        "state": addr.get("state"),
                        "postal_code": addr.get("postal_code"),
                        "country_code": addr.get("country_code"),
                        "telephone_number": addr.get("telephone_number"),
                        "fax_number": addr.get("fax_number")
                    })

                # normalized taxonomy rows
                for t_idx, tax in enumerate(provider.get("taxonomies", []) or [], start=1):
                    taxonomies_rows.append({
                        "query_name": name,
                        "npi": provider.get("number"),
                        "provider_match_index": idx,
                        "taxonomy_index": t_idx,
                        "code": tax.get("code"),
                        "desc": tax.get("desc"),
                        "primary": tax.get("primary"),
                        "state": tax.get("state"),
                        "license": tax.get("license")
                    })

                # normalized identifiers
                for id_idx, ident in enumerate(provider.get("identifiers", []) or [], start=1):
                    identifiers_rows.append({
                        "query_name": name,
                        "npi": provider.get("number"),
                        "provider_match_index": idx,
                        "identifier_index": id_idx,
                        "code": ident.get("code"),
                        "value": ident.get("value"),
                        "state": ident.get("state"),
                        "issuer": ident.get("issuer")
                    })

    # Write outputs
    def write_json(path: str, obj: Any):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)

    def write_csv(path: str, rows: List[Dict[str, Any]]):
        if not rows:
            print(f"Skipping {path} (no rows).")
            return
        fieldnames = list(rows[0].keys())
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(rows)

    write_json("npi_raw.json", raw_out)
    write_csv("providers.csv", providers_rows)
    write_csv("addresses.csv", addresses_rows)
    write_csv("taxonomies.csv", taxonomies_rows)
    write_csv("identifiers.csv", identifiers_rows)

    print("Done.")
    print(f"Names processed: {len(raw_out)}")
    print(f"Providers rows: {len(providers_rows)}")
    print(f"Addresses rows: {len(addresses_rows)}")
    print(f"Taxonomies rows: {len(taxonomies_rows)}")
    print(f"Identifiers rows: {len(identifiers_rows)}")
    print("Files written: npi_raw.json, providers.csv, addresses.csv, taxonomies.csv, identifiers.csv")

if __name__ == "__main__":
    main()


In [0]:
%python
import pandas as pd

# Read providers + addresses
p = pd.read_csv("providers.csv", dtype=str)
a = pd.read_csv("addresses.csv", dtype=str)

# Keep only LOCATION addresses (practice location)
a_loc = a[a["address_purpose"] == "LOCATION"]

# Merge providers with their location address
final_df = p.merge(
    a_loc,
    on="npi",
    how="left",
    suffixes=("", "_addr")
)

# Select a clean, simple set of columns
final_df = final_df[[
    "query_name",
    "npi",
    "first_name",
    "last_name",
    "credential",
    "gender",
    "status",
    "primary_taxonomy_desc",
    "city",
    "state",
    "postal_code",
    "telephone_number"
]]

# Rename columns for clarity
final_df = final_df.rename(columns={
    "primary_taxonomy_desc": "specialty",
    "telephone_number": "phone"
})

# Show everything
display(final_df)

# (Optional) Save to a single CSV
final_df.to_csv("hcp_npi_full_output.csv", index=False)
print("Saved hcp_npi_full_output.csv")
